# Corrected explainability analysis (Grad-CAM, Grad-CAM++, attention) with deletion-based faithfulness
Re-uses the three trained models (unified_results) and the **same shared LC25000 test split**.
Fixes the Grad-CAM++ weighting (standard closed form of Chattopadhay et al., 2018) and adds:
deletion and insertion faithfulness tests (blurred baseline), agreement between methods (Spearman),
and the model-parameter randomisation sanity check (Adebayo et al., 2018).
**Run:** Runtime → T4 GPU → Run all (about 15–20 min). Upload **unified_results.zip** if asked.
At the end it prints a **SUMMARY** and downloads **xai_results.zip**.

### Settings

In [ ]:
import os
TEST_MODE = os.environ.get("XAI_TEST_MODE") == "1"
SEED, IMG_SIZE = 42, 160
N_PER_CLASS = 50          # correctly classified test images per class (as in the original analysis)
N_FIG = 5                 # images per class in the figure
RESULTS_DIR, OUT_DIR = "/content/unified_results", "/content/xai_results"
DATA_DIR = None           # None = download LC25000 via kagglehub
if TEST_MODE:
    RESULTS_DIR, OUT_DIR, DATA_DIR = os.environ["XAI_RESULTS"], os.environ["XAI_OUT"], os.environ["XAI_DATA"]
    N_PER_CLASS, N_FIG = 3, 2

### Imports

In [ ]:
import json, glob, shutil, zipfile, hashlib
import numpy as np, pandas as pd
import matplotlib
if TEST_MODE: matplotlib.use("Agg")
import matplotlib.pyplot as plt
import tensorflow as tf, keras
from keras import layers
from scipy.stats import wilcoxon
os.makedirs(OUT_DIR, exist_ok=True)
CLASSES = ["lung_aca", "lung_n", "lung_scc"]
NAMES = ["MobileNetV2", "EfficientNetB0", "Hybrid_EffNetB0_Transformer"]
SHORT = {"MobileNetV2": "MobileNetV2", "EfficientNetB0": "EfficientNetB0", "Hybrid_EffNetB0_Transformer": "Hybrid"}

### Trained models and test split

In [ ]:
def have(d): return all(os.path.exists(os.path.join(d, n, "best.weights.h5")) for n in NAMES) and os.path.exists(os.path.join(d, "split_test.csv"))
if not have(RESULTS_DIR):
    from google.colab import files
    print("Please choose unified_results.zip:"); up = files.upload()
    os.makedirs(RESULTS_DIR, exist_ok=True); zipfile.ZipFile([k for k in up if k.endswith(".zip")][0]).extractall(RESULTS_DIR)
    if not have(RESULTS_DIR):
        RESULTS_DIR = os.path.dirname(glob.glob(RESULTS_DIR + "/**/split_test.csv", recursive=True)[0])
assert have(RESULTS_DIR)

def find_lung_dir(root):
    for d, ds, _ in os.walk(root):
        if all(c in ds for c in CLASSES): return d
if DATA_DIR is None:
    import kagglehub
    DATA_DIR = find_lung_dir(kagglehub.dataset_download("andrewmvd/lung-and-colon-cancer-histopathological-images"))
test = pd.read_csv(os.path.join(RESULTS_DIR, "split_test.csv"))
test["local"] = [os.path.join(DATA_DIR, c, os.path.basename(p)) for c, p in zip(test["cls"], test["path"])]
ok = [hashlib.md5(open(f, "rb").read()).hexdigest() == m for f, m in zip(test["local"], test["md5"])]
assert all(ok), "test images do not match the original split"
print("Shared test split restored:", len(test), "images (MD5 verified)")

def load(path):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    return tf.cast(tf.round(tf.image.resize(img, (IMG_SIZE, IMG_SIZE))), tf.uint8).numpy().astype("float32")
X = np.stack([load(f) for f in test["local"]]); y = test["label"].values

### Models (identical architectures)

In [ ]:
def head(x):
    x = layers.Dropout(0.3, name="dropout")(x)
    return layers.Activation("softmax", name="pred")(layers.Dense(3, name="logits")(x))
def build(name):
    inp = keras.Input((IMG_SIZE, IMG_SIZE, 3), name="image"); attn = None
    if name == "MobileNetV2":
        x = layers.Rescaling(1 / 127.5, offset=-1, name="mnv2_preprocess")(inp)
        base = keras.applications.MobileNetV2(include_top=False, weights=None, input_tensor=x)
        out = head(layers.GlobalAveragePooling2D(name="gap")(base.output)); cam = "out_relu"
    else:
        base = keras.applications.EfficientNetB0(include_top=False, weights=None, input_tensor=inp); cam = "top_activation"
        if name == "EfficientNetB0":
            out = head(layers.GlobalAveragePooling2D(name="gap")(base.output))
        else:
            f = base.output; h, w, c = f.shape[1], f.shape[2], f.shape[3]
            tok = layers.Reshape((h * w, c), name="tokens")(f); t = layers.LayerNormalization(epsilon=1e-6, name="ln1")(tok)
            a, attn = layers.MultiHeadAttention(num_heads=2, key_dim=16, name="mhsa")(t, t, return_attention_scores=True)
            x = layers.Add(name="res1")([tok, a]); t = layers.LayerNormalization(epsilon=1e-6, name="ln2")(x)
            t = layers.Dense(c, name="mlp2")(layers.Dense(128, activation="gelu", name="mlp1")(t))
            out = head(layers.GlobalAveragePooling1D(name="gap")(layers.Add(name="res2")([x, t])))
    m = keras.Model(inp, out, name=name); m.load_weights(os.path.join(RESULTS_DIR, name, "best.weights.h5"))
    grad_model = keras.Model(m.inputs, [m.get_layer(cam).output, m.get_layer("logits").output])
    attn_model = keras.Model(m.inputs, attn) if attn is not None else None
    return m, grad_model, attn_model

def build_random(name):
    """Same architecture with untrained (random) weights, for the model-randomisation sanity check."""
    global RESULTS_DIR
    saved = keras.Model.load_weights
    keras.Model.load_weights = lambda self, *a, **k: None      # skip loading trained weights
    try: return build(name)
    finally: keras.Model.load_weights = saved

### Saliency methods

In [ ]:
def gradcam_and_pp(grad_model, img, c):
    with tf.GradientTape() as tape:
        A, logits = grad_model(img[None], training=False); S = logits[:, c]
    g = tape.gradient(S, A)[0].numpy().astype("float64"); A = A[0].numpy().astype("float64")
    # Grad-CAM: channel weights = spatial mean of gradients, followed by ReLU
    gc = np.maximum((A * g.mean((0, 1))).sum(-1), 0)
    # Grad-CAM++ (closed form, Chattopadhay et al. 2018, Eq. 19): alpha = g^2 / (2 g^2 + sum(A) g^3)
    g2, g3 = g ** 2, g ** 3
    denom = 2 * g2 + A.sum((0, 1), keepdims=True) * g3
    alpha = np.where(denom != 0, g2 / np.where(denom != 0, denom, 1), 0)
    w = (alpha * np.maximum(g, 0)).sum((0, 1))
    gpp = np.maximum((A * w).sum(-1), 0)
    return gc, gpp
def attention_map(attn_model, img):
    s = attn_model(img[None], training=False).numpy()[0]       # heads x queries x keys
    r = s.mean(0).mean(0); n = int(np.sqrt(r.size)); return r.reshape(n, n)

### Faithfulness: deletion and insertion with a blurred baseline (25 grid cells of 32 x 32 px)

In [ ]:
from scipy.ndimage import gaussian_filter
from scipy.stats import spearmanr
GRID = 5; CELL = IMG_SIZE // GRID; N_RANDOM = 5
def blurred(img): return np.stack([gaussian_filter(img[..., ch], sigma=8) for ch in range(3)], -1)
def del_ins_curves(model, img, c, order):
    """Deletion: replace cells (in order) by the blurred image. Insertion: start from the blurred image and restore cells."""
    base = blurred(img); dele = [img.copy()]; ins = [base.copy()]; cd, ci = img.copy(), base.copy()
    for idx in order:
        r, k = divmod(int(idx), GRID); sl = (slice(r*CELL, (r+1)*CELL), slice(k*CELL, (k+1)*CELL))
        cd = cd.copy(); cd[sl] = base[sl]; dele.append(cd)
        ci = ci.copy(); ci[sl] = img[sl]; ins.append(ci)
    p = model(np.stack(dele + ins), training=False).numpy()[:, c]
    p0 = max(p[0], 1e-8); n = GRID * GRID + 1
    return p[:n] / p0, p[n:] / p0          # both normalised to the unmasked probability
def auc(curve):
    c = np.asarray(curve, float); return float((c.sum() - 0.5 * (c[0] + c[-1])) / (len(c) - 1))   # trapezoidal rule on [0, 1]
def holm(pvals):
    order = np.argsort(pvals); m = len(pvals); adj = np.empty(m); run = 0.0
    for r, i in enumerate(order):
        run = max(run, (m - r) * pvals[i]); adj[i] = min(1.0, run)
    return adj.tolist()

### Run (saliency statistics, faithfulness, agreement between methods, and model-randomisation sanity check)

In [ ]:
rng = np.random.default_rng(SEED)
stats, rows = {}, []
for name in NAMES:
    model, gm, am = build(name)
    keras.utils.set_random_seed(SEED)
    rand_model, rand_gm, rand_am = build_random(name)      # same architecture, untrained random weights
    yp = model.predict(X, batch_size=64, verbose=0).argmax(1)
    sel = np.concatenate([np.where((y == i) & (yp == i))[0][:N_PER_CLASS] for i in range(3)])
    meths = ["Grad-CAM", "Grad-CAM++"] + (["Attention"] if am is not None else [])
    R = {m: {"peak": [], "blank": 0, "del": [], "ins": [], "rand_rho": []} for m in meths}
    R["Random"] = {"del": [], "ins": []}; agree = {}
    for k in sel:
        img, c = X[k], int(y[k])
        gc, gpp = gradcam_and_pp(gm, img, c); maps = {"Grad-CAM": gc, "Grad-CAM++": gpp}
        rgc, rgpp = gradcam_and_pp(rand_gm, img, c); rmaps = {"Grad-CAM": rgc, "Grad-CAM++": rgpp}
        if am is not None: maps["Attention"] = attention_map(am, img); rmaps["Attention"] = attention_map(rand_am, img)
        row = {"model": SHORT[name], "index": int(k), "label": c}
        for m in meths:
            mp = maps[m]; R[m]["peak"].append(int(np.argmax(mp))); R[m]["blank"] += int(mp.max() == 0)
            d, i = del_ins_curves(model, img, c, np.argsort(-mp.ravel(), kind="stable"))
            R[m]["del"].append(auc(d)); R[m]["ins"].append(auc(i))
            rho = spearmanr(mp.ravel(), rmaps[m].ravel()).correlation
            R[m]["rand_rho"].append(0.0 if np.isnan(rho) else float(rho))
            row.update({f"{m}_del": R[m]["del"][-1], f"{m}_ins": R[m]["ins"][-1], f"{m}_rand_rho": R[m]["rand_rho"][-1]})
        dr, ir = zip(*[del_ins_curves(model, img, c, rng.permutation(GRID * GRID)) for _ in range(N_RANDOM)])
        R["Random"]["del"].append(np.mean([auc(x) for x in dr])); R["Random"]["ins"].append(np.mean([auc(x) for x in ir]))
        row.update({"Random_del": R["Random"]["del"][-1], "Random_ins": R["Random"]["ins"][-1]})
        for a_ in range(len(meths)):
            for b_ in range(a_ + 1, len(meths)):
                rho = spearmanr(maps[meths[a_]].ravel(), maps[meths[b_]].ravel()).correlation
                agree.setdefault(f"{meths[a_]} vs {meths[b_]}", []).append(0.0 if np.isnan(rho) else float(rho))
        rows.append(row)
    st = {"images": int(len(sel))}; pv, keys = [], []
    for m in meths:
        cnt = np.bincount(R[m]["peak"], minlength=GRID * GRID); pr = cnt[cnt > 0] / cnt.sum()
        st[m] = {"blank_maps": R[m]["blank"], "distinct_peak_positions": int((cnt > 0).sum()),
                 "most_common_peak_share": float(cnt.max() / len(sel)),
                 "peak_entropy_normalised": float(-(pr * np.log(pr)).sum() / np.log(GRID * GRID)),
                 "deletion_auc_mean": float(np.mean(R[m]["del"])), "deletion_auc_sd": float(np.std(R[m]["del"])),
                 "insertion_auc_mean": float(np.mean(R[m]["ins"])), "insertion_auc_sd": float(np.std(R[m]["ins"])),
                 "randomisation_spearman_mean": float(np.mean(R[m]["rand_rho"])),
                 "randomisation_spearman_sd": float(np.std(R[m]["rand_rho"]))}
        if len(sel) > 5:
            for kind in ["del", "ins"]:
                alt = "less" if kind == "del" else "greater"          # better than random: lower deletion, higher insertion
                pv.append(float(wilcoxon(R[m][kind], R["Random"][kind], alternative=alt).pvalue)); keys.append((m, kind))
    for (m, kind), p, padj in zip(keys, pv, holm(pv) if pv else []):
        st[m][f"{kind}_wilcoxon_p_vs_random"] = p; st[m][f"{kind}_wilcoxon_p_holm"] = padj
    st["Random"] = {"deletion_auc_mean": float(np.mean(R["Random"]["del"])), "deletion_auc_sd": float(np.std(R["Random"]["del"])),
                    "insertion_auc_mean": float(np.mean(R["Random"]["ins"])), "insertion_auc_sd": float(np.std(R["Random"]["ins"]))}
    st["agreement_spearman"] = {k: {"mean": float(np.mean(v)), "sd": float(np.std(v))} for k, v in agree.items()}
    stats[SHORT[name]] = st; print(SHORT[name], json.dumps(st, indent=1))
    # figure: N_FIG correctly classified images per class
    cols = 1 + len(meths); figsel = np.concatenate([np.where((y == i) & (yp == i))[0][:N_FIG] for i in range(3)])
    fig, axes = plt.subplots(len(figsel), cols, figsize=(2.6 * cols, 2.6 * len(figsel)), squeeze=False)
    for r, k in enumerate(figsel):
        img, c = X[k], int(y[k]); gc, gpp = gradcam_and_pp(gm, img, c); maps = {"Grad-CAM": gc, "Grad-CAM++": gpp}
        if am is not None: maps["Attention"] = attention_map(am, img)
        axes[r, 0].imshow(img.astype("uint8")); axes[r, 0].set_title(CLASSES[c], fontsize=8); axes[r, 0].axis("off")
        for j, m in enumerate(meths, 1):
            mm = maps[m]; mm = (mm - mm.min()) / (mm.max() - mm.min() + 1e-12)
            axes[r, j].imshow(img.astype("uint8")); axes[r, j].imshow(tf.image.resize(mm[..., None], (IMG_SIZE, IMG_SIZE)).numpy()[..., 0], cmap="jet", alpha=0.4)
            axes[r, j].set_title(m, fontsize=8); axes[r, j].axis("off")
    fig.tight_layout(); fig.savefig(os.path.join(OUT_DIR, f"fig_xai_{name}.png"), dpi=200, bbox_inches="tight")
    plt.show() if not TEST_MODE else None; plt.close(fig)
    del model, gm, am, rand_model, rand_gm, rand_am; keras.backend.clear_session()

### Summary

In [ ]:
pd.DataFrame(rows).to_csv(os.path.join(OUT_DIR, "xai_per_image.csv"), index=False)
json.dump(stats, open(os.path.join(OUT_DIR, "xai_summary.json"), "w"), indent=1)
shutil.make_archive(OUT_DIR.rstrip("/"), "zip", OUT_DIR)
print("\n" + "=" * 70 + "\nSUMMARY — copy everything below and send it to Claude\n" + "=" * 70)
print(json.dumps(stats, indent=1))
if not TEST_MODE:
    try:
        from google.colab import files; files.download(OUT_DIR.rstrip("/") + ".zip")
    except Exception: print("Download xai_results.zip from the Files panel.")